In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB,BernoulliNB,MultinomialNB,CategoricalNB
from sklearn.ensemble import AdaBoostClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

In [28]:
df1 = pd.read_csv("gender_submission.csv")
df2 = pd.read_csv("train.csv")
df3 = pd.read_csv("test.csv")

In [29]:
print(df1.shape)
print(df2.shape)
print(df3.shape)

(418, 2)
(891, 12)
(418, 11)


In [30]:
df1.head(1)

,PassengerId,Survived
0,892,0


In [31]:
df1.isna().sum()

PassengerId    0
Survived       0
dtype: int64

In [32]:
df1.duplicated().sum()

np.int64(0)

In [33]:
df2.head(1)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S


# Data Processing

In [34]:
df2 = df2.drop(columns=['PassengerId', 'Name', 'Ticket','Cabin'], axis=1)

In [35]:
df2.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [36]:
df2['Age'] = df2['Age'].fillna(df2['Age'].mean())

In [37]:
df2['Embarked'].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [38]:
df3.head(1)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q


In [39]:
df3.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [40]:
df3 = df3.drop(columns=['PassengerId', 'Name', 'Ticket','Cabin'], axis=1)

In [41]:
df3['Age'] = df3['Age'].fillna(df3['Age'].mean())

In [42]:
df3.isna().sum()

Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        1
Embarked    0
dtype: int64

In [43]:
df3['Fare'] = df3['Fare'].fillna(df3['Fare'].mean())

In [44]:
x_train = df2.drop(columns = ['Survived'], axis = 1)
y_train = df2['Survived']

x_test = df3
y_test = df1['Survived']

In [45]:
label = LabelEncoder()

In [46]:
x_train['Sex'] = label.fit_transform(x_train['Sex'])
x_test['Sex'] = label.transform(x_test['Sex'])

In [47]:
x_train['Embarked'] = label.fit_transform(x_train['Embarked'])
x_test['Embarked'] = label.transform(x_test['Embarked'])

In [48]:
std = StandardScaler()

In [49]:
x_train = std.fit_transform(x_train)
x_test = std.transform(x_test)

# Logistic Regression

In [50]:
log_reg = LogisticRegression()

In [51]:
model1 = log_reg.fit(x_train, y_train)

In [52]:
model1.score(x_train, y_train)

0.8013468013468014

In [53]:
y_pred1 = model1.predict(x_test)

In [54]:
accuracy_score(y_pred1, y_test)

0.9473684210526315

In [55]:
# parameter tuning
log_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [100, 200, 500]
}


In [ ]:
log_grid = GridSearchCV(
    log_reg,
    log_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [57]:
log_grid.fit(x_train, y_train)

,estimator,LogisticRegression()
,param_grid,"{'C': [0.01, 0.1, ...], 'max_iter': [100, 200, ...], 'penalty': ['l1', 'l2'], 'solver': ['liblinear', 'saga']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [58]:
log_grid.score(x_train, y_train)

0.8080808080808081

In [59]:
y_pred1 = log_grid.predict(x_test)

In [60]:
accuracy_score(y_pred1, y_test)

0.9473684210526315

# KNN

In [61]:
knn = KNeighborsClassifier()

In [62]:
model2 = knn.fit(x_train, y_train)

In [63]:
model2.score(x_train, y_train)

0.8574635241301908

In [64]:
y_pred2 = model2.predict(x_test)

In [65]:
accuracy_score(y_pred2, y_test)

0.8588516746411483

In [71]:
knn_params = {
    'n_neighbors': [3,5,7,9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
    'p': [1,2]
}

knn_grid = GridSearchCV(
    estimator=knn,
    param_grid=knn_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [72]:
knn_grid.fit(x_train, y_train)

,estimator,KNeighborsClassifier()
,param_grid,"{'metric': ['euclidean', 'manhattan'], 'n_neighbors': [3, 5, ...], 'p': [1, 2], 'weights': ['uniform', 'distance']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,9


In [73]:
knn_grid.score(x_train, y_train)

0.8473625140291807

# Decision Tree

In [75]:
tree = DecisionTreeClassifier()

In [76]:
model3 = tree.fit(x_train, y_train)

In [77]:
model3.score(x_train, y_train)

0.9820426487093153

In [78]:
y_pred3 = model3.predict(x_test)

In [79]:
accuracy_score(y_pred3, y_test)

0.7822966507177034

In [80]:
dt_params = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

In [81]:
tree_grid = GridSearchCV(
    estimator=tree,
    param_grid=dt_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [82]:
tree_grid.fit(x_train, y_train)

,estimator,DecisionTreeClassifier()
,param_grid,"{'criterion': ['gini', 'entropy', ...], 'max_depth': [None, 5, ...], 'max_features': [None, 'sqrt', ...], 'min_samples_leaf': [1, 2, ...], ...}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'log_loss'


In [83]:
tree_grid.score(x_train, y_train)

0.8451178451178452

# AdaBoost

In [84]:
AdaB = AdaBoostClassifier()

In [85]:
AdaB.fit(x_train, y_train)

,estimator,None
,n_estimators,50
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,None


In [86]:
AdaB.score(x_train, y_train)

0.819304152637486

In [87]:
ada_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1],
    'algorithm': ['SAMME']
}

In [88]:
ada_grid = GridSearchCV(
    estimator=AdaB,
    param_grid=ada_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [89]:
ada_grid.fit(x_train , y_train)

c:\Users\ABDULLAH AL MASUM\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


,estimator,AdaBoostClassifier()
,param_grid,"{'algorithm': ['SAMME'], 'learning_rate': [0.01, 0.1, ...], 'n_estimators': [50, 100, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,estimator,None


In [90]:
ada_grid.score(x_train, y_train)

0.819304152637486

# XGBoost

In [91]:
xgb = XGBClassifier()

In [92]:
xgb.fit(x_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [93]:
xgb.score(x_train, y_train)

0.9640852974186308

In [95]:
xgb.score(x_test, y_test)

0.8181818181818182

In [94]:
y_pred4 = xgb.predict(x_test)

In [96]:
accuracy_score(y_test, y_pred4)

0.8181818181818182

In [97]:
xgb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 0.1, 0.2]
}

In [98]:
xgb_grid = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [99]:
xgb_grid.fit(x_train, y_train)

,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'colsample_bytree': [0.8, 1.0], 'gamma': [0, 0.1, ...], 'learning_rate': [0.01, 0.1, ...], 'max_depth': [3, 5, ...], ...}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [100]:
xgb.score(x_train, y_train)

0.9640852974186308

In [101]:
xgb.score(x_test, y_test)

0.8181818181818182

# SVC

In [102]:
svc = SVC()

In [103]:
svc.fit(x_train, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [104]:
svc.score(x_train, y_train)

0.8395061728395061

In [105]:
svc.score(x_test, y_test)

0.930622009569378

In [106]:
svc_params = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto', 0.1, 0.01]
}

In [107]:
svc_grid = GridSearchCV(
    estimator=svc,
    param_grid=svc_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [108]:
svc.fit(x_train, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [109]:
svc.score(x_train, y_train)

0.8395061728395061

# Bernoulli Naive Bayes

In [110]:
BNB = BernoulliNB()

In [111]:
BNB.fit(x_train, y_train)

,alpha,1.0
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [112]:
BNB.score(x_train, y_train)

0.7710437710437711

In [113]:
bnb_params = {
    'alpha': [0.1, 0.5, 1.0, 2.0],
    'binarize': [0.0, 0.5, 1.0],
    'fit_prior': [True, False]
}

In [114]:
bnb_grid = GridSearchCV(
    estimator=BNB,
    param_grid=bnb_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [115]:
bnb_grid.fit(x_train, y_train)

,estimator,BernoulliNB()
,param_grid,"{'alpha': [0.1, 0.5, ...], 'binarize': [0.0, 0.5, ...], 'fit_prior': [True, False]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,0.1


In [116]:
bnb_grid.score(x_train, y_train)

0.7732884399551067

# KNN best 

In [117]:
y_pred2

array([0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1,
       1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [119]:
df1 = df1.drop(columns=['Survived'], axis=1)

In [120]:
df1.head()

,PassengerId
0,892
1,893
2,894
3,895
4,896


In [121]:
pred_df = pd.DataFrame({
  "PassengerId" : df1['PassengerId'],
  "Survived" : y_pred2
})

In [123]:
pred_df

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,1
4,896,0
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [125]:
pred_df.to_csv("submission.csv", index=False)